# Нео-Гук вместо «Гука на тензоре Альманси» в выводе уравнения движения в координатах Эйлера
Дополнение к `Vyvod.ipynb` (3 ноя 2026). Исходный ноутбук не изменён.
**Мотивация.** В `Vyvod.ipynb` напряжение задано линейным законом Гука на тензоре Альманси $\sigma=\lambda\,\mathrm{tr}\eta\,I+2\mu\eta$. Для прямой $\pi$-дисклинации этот закон не дал решения (потеря эллиптичности), а гиперупругий нео-Гук дал (`electron_structure_rings/verification`, разд. 13–14).
Здесь сравниваем оба закона на той же одномерной продольной волне, что и в `Vyvod.ipynb` разд. 2–4.

In [ ]:
import numpy as np
def report(name, ok): print(('OK   ' if ok else 'FAIL ') + name); return ok
mu = 1.0

## 1. Первая Пиолы–Кирхгофа $P(F)$ для 1D-растяжения $F=1+U_X$
- Гук на Альманси (как в Vyvod.ipynb): $\sigma=(\lambda+2\mu)\eta$, $\eta=\tfrac12(1-F^{-2})$; в 1D $P=\sigma$ (площадь не меняется) $\Rightarrow P_A=\tfrac{\lambda+2\mu}{2}(1-F^{-2})$.
- Нео-Гук: $W=\tfrac\mu2(F^2-1-2\ln F)+\tfrac\lambda2(\ln F)^2\Rightarrow P_N=\mu(F-F^{-1})+\lambda\ln F/F$.

Разложение $P=(\lambda+2\mu)\big[e-\beta e^2+\dots\big]$, $e=U_X$, даёт коэффициент нелинейности $\beta$ в уравнении $U_{tt}=c^2(1-2\beta U_X)U_{XX}$ из Vyvod.ipynb разд. 4:
$$\beta_A=\tfrac32,\qquad \beta_N=\frac{\mu+\tfrac32\lambda}{\lambda+2\mu}.$$
Сравнение с Ландау–Лифшицем ($\beta=\tfrac32+\tfrac{A+3B+C}{\lambda+2\mu}$): нео-Гук отвечает $A+3B+C=-2\mu$, Гук на Альманси — $A+3B+C=0$.

In [ ]:
for lam in (1., 2., 10.):
    bA, bN = 1.5, (mu+1.5*lam)/(lam+2*mu); e = 1e-4; F = 1+e; M = lam+2*mu
    PA = M/2*(1-1/F**2); PN = mu*(F-1/F) + lam*np.log(F)/F
    report(f'λ/μ={lam:.0f}: β_Альм = {-(PA-M*e)/(M*e*e):.4f} ≈ 1.5, β_нео = {-(PN-M*e)/(M*e*e):.4f} ≈ {bN:.4f}',
           abs(-(PA-M*e)/(M*e*e)-bA) < 1e-3 and abs(-(PN-M*e)/(M*e*e)-bN) < 1e-3)
    report(f'      A+3B+C = (β_нео−3/2)(λ+2μ) = {(bN-1.5)*M:+.3f}μ = −2μ', np.isclose((bN-1.5)*M, -2*mu))

## 2. Эллиптичность (ключевое отличие)
Условие эллиптичности в 1D: $dP/dF>0$. Для Гука на Альманси $dP_A/dF=(\lambda+2\mu)F^{-3}>0$, но $P_A\to-\infty$ при $F\to0$ медленно, а **в 2D/3D** (дисклинация) сдвиговая часть теряет эллиптичность. Для нео-Гука $W\to\infty$ при $F\to0$ и $dP_N/dF=\mu(1+F^{-2})+\lambda(1-\ln F)/F^2$.
Проверяем $dP_N/dF>0$ на $F\in(0.05, 60)$ и что энергия, запасённая при сжатии $F\to0$, бесконечна (нельзя схлопнуть объём).

In [ ]:
Fs = np.linspace(.05, 60, 200000)
for lam in (1., 2., 10.):
    dPN = mu*(1+1/Fs**2) + lam*(1-np.log(Fs))/Fs**2
    report(f'λ/μ={lam:.0f}: min dP_нео/dF = {dPN.min():.3f} > 0', dPN.min() > 0)
WN = lambda F, lam: mu/2*(F**2-1-2*np.log(F)) + lam/2*np.log(F)**2
WA = lambda F, lam: (lam+2*mu)/2*(F - 2 + 1/F) / 1   # ∫P_A dF от 1 до F = (λ+2μ)/2 (F + 1/F − 2)
report(f'нео-Гук: W(F→0) растёт неограниченно ({WN(1e-6, 1.):.1f} при F=1e−6)', WN(1e-6, 1.) > 50)
report(f'Гук на Альманси: W(F→0) тоже растёт, но как 1/F ({WA(1e-6,1.):.1e}); в 1D оба эллиптичны — различие проявляется в 2D', WA(1e-6, 1.) > 50)

## 3. Вторая гармоника продольной волны с нео-Гуком
Решение второго порядка из Vyvod.ipynb разд. 4: $U_2=-\tfrac{\beta k^2A^2}{4}X\cos2\psi$ — меняется только $\beta$. Расстояние образования разрыва $x_s=\dfrac{1}{\beta\,\varepsilon_0 k}$ (для $\varepsilon_0=kA$) растёт в $\beta_A/\beta_N$ раз.

In [ ]:
import numpy as np
def run(beta, A=2e-3, c=1., k=2*np.pi, L=3., N=3000, T=None):
    w = c*k; dx = L/N; dt = .4*dx/c; x = np.arange(N+1)*dx; T = T or .9*L/c
    U = np.zeros(N+1); Up = U.copy(); t = 0.
    while t < T:
        Ux = np.gradient(U, dx); Uxx = np.zeros_like(U); Uxx[1:-1] = (U[2:]-2*U[1:-1]+U[:-2])/dx**2
        Un = 2*U - Up + (c*dt)**2*(1-2*beta*Ux)*Uxx; Un[0] = A*np.sin(w*(t+dt)); Un[-1] = 0
        Up, U = U, Un; t += dt
    return x, U, t, w, k, c
lam = 1.; bN = (mu+1.5*lam)/(lam+2*mu)
for beta in (1.5, bN):
    x, U, t, w, k, c = run(beta); X = 1.5; i = int(X/x[1]); win = slice(i-500, i+500)   # одна длина волны
    ps = w*t - k*x[win]; G = np.stack([np.sin(ps), np.cos(ps), np.sin(2*ps), np.cos(2*ps), np.ones_like(ps)], 1)
    co = np.linalg.lstsq(G, U[win], rcond=None)[0]; amp2 = np.hypot(co[2], co[3])
    report(f'β={beta:.3f}: амплитуда 2-й гармоники при X={X}: {amp2:.2e} ≈ βk²A²X/4 = {beta*k*k*4e-6*X/4:.2e}',
           abs(amp2/(beta*k*k*4e-6*X/4) - 1) < .2)
print(f'x_s(нео)/x_s(Альманси) = β_A/β_N = {1.5/bN:.2f} при λ=μ')

## Итог
| | Гук на Альманси (Vyvod.ipynb) | нео-Гук |
|---|---|---|
| $\beta$ при $\lambda=\mu$ | 3/2 | 5/6 |
| $A+3B+C$ (Ландау–Лифшиц) | 0 | $-2\mu$ |
| прямая $\pi$-дисклинация | решения нет | решение есть, $E/E_{lin}=1.06\,/\,0.85$ |
| поликонвексность | нет | да |

Вывод: эйлерова кинематика из Vyvod.ipynb сохраняется, но определяющий закон для больших поворотов (мёбиус-кольцо) нужно брать поликонвексным; $\beta$ продольной волны тогда зависит от $\lambda/\mu$.